In [0]:
import os
import shutil


key_path = "/Volumes/project_lakehouse/bronze_layer/credentials/gcp-key.json"

try:
  with open(key_path, "r") as f:
    key_content = f.read()
    print("Successfully read the key content")

    spark.conf.set("google.cloud.auth.service.account.enable", "true")
    spark.conf.set("google.cloud.auth.service.account.json.key", key_content)
    print("Spark configured via Direct Injection")
 
except Exception as e:
  print(f"Error: {e}")

Successfully read the key content
Spark configured via Direct Injection


In [0]:
bronze_path = "gs://data-lakehouse-bronze/raw/*/*/*/*.json"

df_bronze = spark.read.json(bronze_path)

print(f"Total records: {df_bronze.count()}")
df_bronze.show()

Total records: 45
+-------+--------------------+--------------------+
|bitcoin|extraction_timestamp|       source_system|
+-------+--------------------+--------------------+
|{67275}|2026-03-08T08:34:...|databricks_automa...|
|{67295}|2026-03-08T08:39:...|databricks_automa...|
|{67731}|2026-03-08T09:38:...|databricks_automa...|
|{68017}|2026-03-08T10:38:...|databricks_automa...|
|{67574}|2026-03-08T11:38:...|databricks_automa...|
|{67308}|2026-03-08T12:38:...|databricks_automa...|
|{67061}|2026-03-08T13:38:...|databricks_automa...|
|{67277}|2026-03-08T14:38:...|databricks_automa...|
|{67148}|2026-03-08T15:38:...|databricks_automa...|
|{66890}|2026-03-08T16:38:...|databricks_automa...|
|{67035}|2026-03-08T17:38:...|databricks_automa...|
|{66896}|2026-03-08T18:38:...|databricks_automa...|
|{67327}|2026-03-08T19:38:...|databricks_automa...|
|{67372}|2026-03-08T20:38:...|databricks_automa...|
|{66971}|2026-03-08T21:38:...|databricks_automa...|
|{66054}|2026-03-08T22:38:...|databricks_autom

In [0]:
table_name = "project_lakehouse.bronze_layer.btc_raw"

df_bronze.write.format('delta').mode('overwrite').saveAsTable(table_name)

print(f"Success raw data is now a permanent Delta table: {table_name}")

Success raw data is now a permanent Delta table: project_lakehouse.bronze_layer.btc_raw


In [0]:
%sql
--Fix the Bronze Raw Table
UPDATE project_lakehouse.bronze_layer.btc_raw
SET source_system = 'databricks_automated_job'
WHERE source_system = 'databricks_autoated_job';

num_affected_rows
20


In [0]:
%sql
SELECT * FROM project_lakehouse.bronze_layer.btc_raw

bitcoin,extraction_timestamp,source_system
List(72320),2026-03-05T20:37:21.008409,coingecko_api
List(71300),2026-03-05T21:01:36.478156,coingecko_api
List(71080),2026-03-05T22:00:12.938210,coingecko_api
List(71172),2026-03-05T22:05:23.045176,coingecko_api
List(67275),2026-03-08T08:34:57.490249,databricks_automated_job
List(67295),2026-03-08T08:39:41.395794,databricks_automated_job
List(67731),2026-03-08T09:38:51.250960,databricks_automated_job
List(68017),2026-03-08T10:38:53.879318,databricks_automated_job
List(67574),2026-03-08T11:38:20.679228,databricks_automated_job
List(67308),2026-03-08T12:38:28.980443,databricks_automated_job


In [0]:
from pyspark.sql.functions import col, to_timestamp

df_bronze_raw = spark.read.table("project_lakehouse.bronze_layer.btc_raw")
 
df_silver = df_bronze_raw.select(col('bitcoin.usd').alias("price_usd"),to_timestamp(col("extraction_timestamp")).alias("event_timestamp"),col("source_system"))

display(df_silver)

price_usd,event_timestamp,source_system
72320,2026-03-05T20:37:21.008409Z,coingecko_api
71300,2026-03-05T21:01:36.478156Z,coingecko_api
71080,2026-03-05T22:00:12.93821Z,coingecko_api
71172,2026-03-05T22:05:23.045176Z,coingecko_api
67275,2026-03-08T08:34:57.490249Z,databricks_automated_job
67295,2026-03-08T08:39:41.395794Z,databricks_automated_job
67731,2026-03-08T09:38:51.25096Z,databricks_automated_job
68017,2026-03-08T10:38:53.879318Z,databricks_automated_job
67574,2026-03-08T11:38:20.679228Z,databricks_automated_job
67308,2026-03-08T12:38:28.980443Z,databricks_automated_job


In [0]:
%sql
create schema if not exists project_lakehouse.silver_layer;

In [0]:
from delta.tables import DeltaTable

target_table_name = "project_lakehouse.silver_layer.btc_prices_clean"

if not spark.catalog.tableExists(target_table_name):
  df_silver.write.format("delta").saveAsTable(target_table_name)
  print("Created Silver table for the first time")
else:
    target_table = DeltaTable.forName(spark, target_table_name)
    
    target_table.alias("target")\
        .merge(df_silver.alias("source"),
        "target.event_timestamp = source.event_timestamp"
        )\
        .whenNotMatchedInsertAll()\
        .execute()
    print("Merge completed")

Merge completed


In [0]:
%sql
create schema if not exists project_lakehouse.gold_layer

In [0]:
%sql
create or replace table project_lakehouse.gold_layer.btc_price_trends as 

with price_lagged as (
  select event_timestamp, price_usd,
    lag(price_usd) over (order by event_timestamp) as previous_price
  from project_lakehouse.silver_layer.btc_prices_clean
)

select event_timestamp, price_usd, previous_price, (price_usd - previous_price) as price_difference,
round(((price_usd - previous_price)/previous_price) * 100, 4) as percentage_change
from price_lagged;


num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from project_lakehouse.gold_layer.btc_price_trends;

event_timestamp,price_usd,previous_price,price_difference,percentage_change
2026-03-05T20:37:21.008409Z,72320,null,null,null
2026-03-05T21:01:36.478156Z,71300,72320,-1020,-1.4104
2026-03-05T22:00:12.93821Z,71080,71300,-220,-0.3086
2026-03-05T22:05:23.045176Z,71172,71080,92,0.1294
2026-03-07T12:41:49.007841Z,67997,71172,-3175,-4.461
2026-03-07T13:38:53.406007Z,67997,67997,0,0.0
2026-03-07T14:38:30.37724Z,67833,67997,-164,-0.2412
2026-03-07T15:37:54.144202Z,68006,67833,173,0.255
2026-03-07T16:38:50.558105Z,67791,68006,-215,-0.3161
2026-03-07T17:38:55.270732Z,67874,67791,83,0.1224


Databricks visualization. Run in Databricks to view.